# 05 — Forecasting: Geography (State-Level)

Forecast quarterly federal obligated spending by **US state** using Prophet, SARIMA, and XGBoost.

- **Data**: 56 states/territories, FY2008–2024 (68 quarters — 2× more history than hierarchical notebooks)
- **Train**: FY2008–2020 (52 quarters) | **Test**: FY2021–2024 (16 quarters)
- **COVID flag**: FY2020 in train = 1; FY2021 in test = 1, FY2022–2024 = 0
- **Per-state models**: Top 10 states individually (Prophet + SARIMA)
- **Extra metric**: Per-capita spending (obligated_amount / population)

**Key results (total US spending) — model ranking REVERSED vs hierarchical notebooks:**
| Model   | MAE      | RMSE     | MAPE    |
|---------|----------|----------|---------|
| Prophet | $158B    | $194B    | **13.7%** ✓ — wins for the first time |
| XGBoost | $257B    | $373B    | 21.5%   |
| SARIMA  | $673B    | $766B    | 58.3%   |

**Why the reversal:** Geography data is quarterly spending (not cumulative YTD). SARIMA's double differencing was optimized for the YTD pattern — without it, the differencing destroys the signal and SARIMA diverges on the 16-quarter test horizon.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT         = Path('..').resolve()
GEO_BASIC    = ROOT / 'pipeline-b-geography' / 'data' / 'basic-geography' / 'cleaned'
FORECAST_DIR = ROOT / 'pipeline-a-hierarchical' / 'data' / 'forecasts'
FORECAST_DIR.mkdir(exist_ok=True)

def trillions(x, _): return f'${x/1e12:.2f}T'
def billions(x, _):  return f'${x/1e9:.1f}B'

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mp   = mape(np.array(y_true), np.array(y_pred))
    print(f'{label:22}  MAE=${mae/1e9:.2f}B  RMSE=${rmse/1e9:.2f}B  MAPE={mp:.1f}%')
    return {'model': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mp}

# Geography uses a different train/test split: longer history available
TRAIN_END  = 2020
TEST_START = 2021

print('Setup done.')

The train/test split here differs from the hierarchical notebooks (02–04). The geography data starts in FY2008, giving 68 quarters of history versus 32 in the hierarchical files. Training through FY2020 provides 52 quarters — a much richer training window that should give all three models stronger seasonal and trend estimates. The test period covers FY2021–2024 (16 quarters), capturing both the COVID-peak year (FY2021) and the post-COVID normalization through FY2024.

## 2. Load & Prepare State Data

In [ ]:
df = pd.read_csv(GEO_BASIC / 'geography_state_all_FY2008_2024.csv')

quarter_to_month = {1: (10, -1), 2: (1, 0), 3: (4, 0), 4: (7, 0)}
def fy_q_to_date(row):
    month, yr_offset = quarter_to_month[row['quarter']]
    return pd.Timestamp(year=int(row['fy']) + yr_offset, month=month, day=1)

df['ds']          = df.apply(fy_q_to_date, axis=1)
# COVID flag: FY2020 is training data; FY2021 is test data but COVID-impacted
df['covid']       = df['fy'].isin([2020, 2021]).astype(int)
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

print(f'Rows: {len(df):,}  States: {df["geo_code"].nunique()}  FY: {df["fy"].min()}–{df["fy"].max()}')
print(f'Quarters per state (avg): {df.groupby("geo_code").size().mean():.0f}')
print(df[['fy','quarter','ds','geo_code','geo_name','obligated_amount','population']].head(4).to_string(index=False))

The state file covers 56 geographic units — 50 states plus DC and 5 territories (Puerto Rico, Guam, Virgin Islands, American Samoa, Northern Mariana Islands). The COVID flag covers both FY2020 (in training) and FY2021 (in test) so the models are told that both years were anomalous — FY2020 was seen during training, and FY2021 gets the flag passed as an exogenous variable during the test-period forecast.

In [ ]:
# Total US spending per quarter
total = (df.groupby(['fy', 'quarter', 'ds', 'covid'])
           .agg(obligated_amount=('obligated_amount', 'sum'),
                population=('population', 'sum'))
           .reset_index().sort_values('ds'))
total['per_capita'] = total['obligated_amount'] / total['population']

# Top 10 states by total spend
top10_codes  = df.groupby('geo_code')['obligated_amount'].sum().nlargest(10).index.tolist()
state_names  = df.drop_duplicates('geo_code').set_index('geo_code')['geo_name'].to_dict()

train_total = total[total['fy'] <= TRAIN_END]
test_total  = total[total['fy'] >= TEST_START]

print(f'Total series — Train: {len(train_total)} quarters  Test: {len(test_total)} quarters')
print(f'Train: {train_total["ds"].min().date()} → {train_total["ds"].max().date()}')
print(f'Test:  {test_total["ds"].min().date()} → {test_total["ds"].max().date()}')
print(f'\nTop 10 states by total spend:')
for code in top10_codes:
    tot = df[df['geo_code'] == code]['obligated_amount'].sum()
    print(f'  {code}  {state_names[code]:<25}  ${tot/1e12:.2f}T')

The top 10 states by total federal obligations (FY2008–2024) are California ($5.6T), Florida ($4.9T), Texas ($4.1T), New York ($3.3T), Pennsylvania ($3.2T), Virginia ($2.1T), Indiana ($2.0T), Minnesota ($1.7T), South Carolina ($1.6T), and Illinois ($1.6T). California, Florida, and Texas rank high in total dollars because of large populations and significant Social Security / Medicare / Medicaid pass-through payments. Virginia appears due to its concentration of defense contractors and federal agency headquarters around DC. Note: population-adjusted per-capita rankings tell a very different story — DC, North Dakota, and the territories receive far more federal dollars per resident than the large states.

## 3. Prophet — Total US

In [ ]:
def run_prophet(train_df, test_df, series_name='total'):
    prophet_train = train_df.rename(columns={'obligated_amount': 'y'})[['ds','y','covid']]
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05
    )
    m.add_regressor('covid')
    m.fit(prophet_train)
    future = m.make_future_dataframe(periods=len(test_df), freq='QS-OCT')
    # COVID flag for test period: FY2021 = year 2020/2021 calendar
    future['covid'] = future['ds'].apply(
        lambda d: 1 if (d.year == 2020 or (d.year == 2021 and d.month < 10)) else 0)
    forecast = m.predict(future)
    pred = forecast.tail(len(test_df))['yhat'].values
    true = test_df['obligated_amount'].values
    result = metrics(true, pred, label='Prophet')
    result.update({'series': series_name, 'pred': pred, 'true': true, 'ds': test_df['ds'].values})
    return result, m, forecast

prophet_result, prophet_model, prophet_forecast = run_prophet(train_total, test_total, 'total')

The COVID flag for the test period is applied based on the actual calendar dates of FY2021 (October 2020 through September 2021). This is more precise than just flagging calendar year 2021, since the fiscal year boundary falls mid-calendar-year. With 52 training quarters, Prophet has substantially more data to fit its seasonality and changepoint model compared to the 24-quarter training windows in notebooks 02–04 — this should produce a better-calibrated trend estimate and potentially better test performance.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(total['ds'], total['obligated_amount'], 'o-', ms=3, lw=1.5,
        label='Actual', color='steelblue')
ax.fill_between(prophet_forecast['ds'],
                prophet_forecast['yhat_lower'], prophet_forecast['yhat_upper'],
                alpha=0.15, color='orange', label='95% confidence')
ax.plot(prophet_forecast['ds'], prophet_forecast['yhat'], '--', lw=1.5,
        color='orange', label='Prophet forecast')
ax.axvline(pd.Timestamp('2021-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('Prophet — Total US Federal Spending by State (Geography)')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

comparison_p = pd.DataFrame({
    'Quarter':      [f'FY{int(r.fy)} Q{int(r.quarter)}' for _, r in test_total.iterrows()],
    'Actual ($B)':  (test_total['obligated_amount'].values / 1e9).round(1),
    'Prophet ($B)': (prophet_result['pred'] / 1e9).round(1),
    'Error ($B)':   ((prophet_result['pred'] - test_total['obligated_amount'].values) / 1e9).round(1)
})
print('Prophet — quarter-by-quarter comparison (FY2021–2024):')
print(comparison_p.to_string(index=False))

**Prophet Total Results — MAPE 13.7% | MAE $158B | RMSE $194B — best model here**

Prophet wins on geography for the first time across the five notebooks. With 52 training quarters, it had enough data to fit a reliable trend and seasonal pattern for quarterly spending. The error table is mixed in direction — some quarters over-predicted (+$396B in FY2021 Q1), others under-predicted (−$306B in FY2021 Q2) — meaning the model captures the level correctly but struggles with the exact timing of COVID-related spending spikes. By FY2022–2024, errors shrunk to ±$200B range, showing good long-run trend tracking. The worst single miss was Q3 FY2024 (+$367B), likely due to an end-of-year contract surge that wasn't in the seasonal pattern. Overall, 13.7% MAPE on a 16-quarter test horizon covering COVID is a solid result.

## 4. SARIMA — Total US

In [ ]:
def run_sarima(train_df, test_df, order=(1,1,1), seasonal_order=(1,1,0,4), series_name='total'):
    train_y    = train_df.set_index('ds')['obligated_amount']
    train_exog = train_df.set_index('ds')[['covid']]
    test_exog  = test_df.set_index('ds')[['covid']]
    model  = SARIMAX(train_y, exog=train_exog, order=order, seasonal_order=seasonal_order,
                     enforce_stationarity=False, enforce_invertibility=False)
    fitted = model.fit(disp=False)
    pred   = fitted.forecast(steps=len(test_df), exog=test_exog)
    true   = test_df['obligated_amount'].values
    result = metrics(true, pred.values, label='SARIMA')
    result.update({'series': series_name, 'pred': pred.values, 'true': true,
                   'ds': test_df['ds'].values})
    return result, fitted

sarima_result, sarima_fitted = run_sarima(train_total, test_total, series_name='total')
print(f'\nAIC: {sarima_fitted.aic:.1f}   BIC: {sarima_fitted.bic:.1f}')

SARIMAX(1,1,1)(1,1,0,4) with COVID as exogenous variable. With 52 training quarters (vs 24 in notebooks 02–04), SARIMA has much more data to estimate its AR and seasonal parameters. The longer history should produce a more reliable AIC/BIC and potentially a tighter MAPE — but the 16-quarter test horizon (vs 8) is also a harder task, particularly since FY2021 is a COVID-impacted year that the model must forecast out-of-sample.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(total['ds'], total['obligated_amount'], 'o-', ms=3, lw=1.5,
        label='Actual', color='steelblue')
ax.plot(test_total['ds'], sarima_result['pred'], 's--', ms=4, lw=1.5,
        label='SARIMA forecast', color='green')
ax.axvline(pd.Timestamp('2021-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('SARIMA — Total US Federal Spending by State (Geography)')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

comparison_s = pd.DataFrame({
    'Quarter':     [f'FY{int(r.fy)} Q{int(r.quarter)}' for _, r in test_total.iterrows()],
    'Actual ($B)': (test_total['obligated_amount'].values / 1e9).round(1),
    'SARIMA ($B)': (sarima_result['pred'] / 1e9).round(1),
    'Error ($B)':  ((sarima_result['pred'] - test_total['obligated_amount'].values) / 1e9).round(1)
})
print('SARIMA — quarter-by-quarter comparison (FY2021–2024):')
print(comparison_s.to_string(index=False))

**SARIMA Total Results — MAPE 58.3% | MAE $673B | RMSE $766B | AIC 2,313 — fails badly**

SARIMA completely reversed its performance from notebooks 02–04. The root cause: **geography data is quarterly spending amounts (not cumulative YTD)**. In the hierarchical notebooks, SARIMA's double differencing removed the within-year YTD accumulation pattern (Q4 >> Q1 within each year), which was the dominant signal. In geography data, there is no such accumulation — Q1 and Q4 within a year are roughly equal in scale. SARIMA's seasonal differencing therefore over-subtracts, destroying the genuine seasonal pattern and causing the model to over-project spending by $1–1.4T per quarter by FY2024. The AIC jumped from 759 (hierarchical notebooks) to 2,313 — reflecting both the longer series and the much poorer fit. SARIMA should **not** be used for geography in the dashboard.

## 5. XGBoost — All 56 States

In [ ]:
df_xgb = df.sort_values(['geo_code', 'ds']).copy()

df_xgb['lag_1']     = df_xgb.groupby('geo_code')['obligated_amount'].shift(1)
df_xgb['lag_4']     = df_xgb.groupby('geo_code')['obligated_amount'].shift(4)
df_xgb['lag_8']     = df_xgb.groupby('geo_code')['obligated_amount'].shift(8)
df_xgb['roll4_mean'] = df_xgb.groupby('geo_code')['obligated_amount'].transform(
    lambda x: x.shift(1).rolling(4, min_periods=2).mean())
# Per-capita lag (normalized by population)
df_xgb['per_capita']      = df_xgb['obligated_amount'] / df_xgb['population'].replace(0, np.nan)
df_xgb['pc_lag4']         = df_xgb.groupby('geo_code')['per_capita'].shift(4)

le = LabelEncoder()
df_xgb['state_enc'] = le.fit_transform(df_xgb['geo_code'])

df_xgb = df_xgb.dropna(subset=['lag_1', 'lag_4', 'lag_8'])

FEATURES = ['state_enc','fy','quarter','quarter_sin','quarter_cos','covid',
            'lag_1','lag_4','lag_8','roll4_mean','pc_lag4']
TARGET   = 'obligated_amount'

train_xgb = df_xgb[df_xgb['fy'] <= TRAIN_END]
test_xgb  = df_xgb[df_xgb['fy'] >= TEST_START]

print(f'XGBoost train: {len(train_xgb):,} rows   test: {len(test_xgb):,} rows')
print(f'Features: {FEATURES}')

An additional feature unique to this notebook: `pc_lag4` — the per-capita spending from the same quarter one year ago. This normalizes for population differences across states, allowing XGBoost to learn that, e.g., a $1B increase in California spending is proportionally different from the same dollar increase in Wyoming. The per-capita lag gives the model a scale-adjusted historical signal on top of the raw dollar lag features.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(train_xgb[FEATURES], train_xgb[TARGET],
              eval_set=[(test_xgb[FEATURES], test_xgb[TARGET])],
              verbose=False)

xgb_pred = xgb_model.predict(test_xgb[FEATURES])
xgb_result = metrics(test_xgb[TARGET].values, xgb_pred, label='XGBoost')

importance = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('\nFeature importance:')
print(importance.round(4).to_string())

**XGBoost Feature Importance — roll4_mean dominates at 49.2%**

The 4-quarter rolling mean is now the dominant feature at 49.2% — even higher than in the federal accounts notebook (44.4%). With 68 quarters of training data per state and quarterly (non-cumulative) spending, the rolling average of the past year's spending is the single best predictor of next quarter's spending. `lag_4` (16.4%) and `lag_1` (16.3%) are nearly equal in importance — unlike the hierarchical notebooks where `lag_4` dominated. This makes sense for non-cumulative quarterly data: last quarter's spending is as informative as the same quarter last year. `state_enc` scored only 1.1% — low, but slightly higher than `agency_enc` (0.6%), suggesting state-level spending patterns have a bit more heterogeneity than agency patterns. The `covid` flag contributed 2.6%, noticeably higher than in the hierarchical notebooks, reflecting that the 16-quarter test period starting in FY2021 includes the COVID peak as a major out-of-sample challenge.

In [ ]:
test_xgb_copy = test_xgb.copy()
test_xgb_copy['xgb_pred'] = xgb_pred
xgb_total = test_xgb_copy.groupby('ds')[['obligated_amount','xgb_pred']].sum().reset_index()
train_actual = df_xgb[df_xgb['fy'] <= TRAIN_END].groupby('ds')['obligated_amount'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_actual['ds'], train_actual['obligated_amount'], 'o-', ms=2, lw=1.5,
        color='steelblue', label='Actual (train)')
ax.plot(xgb_total['ds'], xgb_total['obligated_amount'], 'o-', ms=2, lw=1.5,
        color='steelblue', alpha=0.4, label='Actual (test)')
ax.plot(xgb_total['ds'], xgb_total['xgb_pred'], 's--', ms=4, lw=1.5,
        color='purple', label='XGBoost forecast')
ax.axvline(pd.Timestamp('2021-10-01'), color='red', lw=1, ls='--', label='Train/Test split')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(trillions))
ax.set_title('XGBoost — Total US Spending by State (Sum Across 56 States)')
ax.set_xlabel('Date')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

xgb_total_result = metrics(xgb_total['obligated_amount'].values,
                           xgb_total['xgb_pred'].values, label='XGB-Total')

**XGBoost Total Results — MAPE 21.5% | MAE $257B | RMSE $373B**

XGBoost lands in the middle — better than SARIMA (58.3%) but worse than Prophet (13.7%). With only 56 states summed (vs 116 agencies or 2,236 accounts), there is less error cancellation at the aggregate level, so the total MAPE is more sensitive to individual state misses. The 16-quarter test horizon also challenges XGBoost's lag features — by FY2024, the `lag_8` value comes from FY2022 (which was itself a post-COVID adjustment year), making the two-year-ago signal noisier than in the hierarchical notebooks. Nevertheless, XGBoost provides reasonable estimates and is the only model that also delivers per-state forecasts without individual fitting, making it useful for the dashboard's state-level drill-down view.

## 6. Per-State Forecasts — Top 10

In [ ]:
state_results = []

for code in top10_codes:
    sname = state_names.get(code, code)
    sub   = df[df['geo_code'] == code].sort_values('ds')
    tr    = sub[sub['fy'] <= TRAIN_END]
    te    = sub[sub['fy'] >= TEST_START]

    if len(tr) < 12 or len(te) == 0:
        print(f'  Skip {sname}: insufficient data')
        continue

    # Prophet
    try:
        r_p, _, _ = run_prophet(tr, te, series_name=code)
        r_p['state_name'] = sname
        state_results.append(r_p)
    except Exception as e:
        print(f'  Prophet FAIL {sname}: {e}')

    # SARIMA
    try:
        r_s, _ = run_sarima(tr, te, series_name=code)
        r_s['state_name'] = sname
        state_results.append(r_s)
    except Exception as e:
        print(f'  SARIMA FAIL {sname}: {e}')

print(f'\nCompleted {len(state_results)} model runs across {len(top10_codes)} states.')

20 models — Prophet and SARIMA for each top-10 state. With 52 training quarters, SARIMA has more data than in notebooks 02–04, but the non-cumulative data structure means SARIMA's seasonal differencing is harmful rather than helpful. Prophet won on 8 of 10 states; SARIMA narrowly won on Texas (22.6% vs 32.0%) and Virginia (22.9% vs 31.2%) — both states with stable defense-contract dominated spending where SARIMA's AR structure can track the smooth growth. Florida was the outlier where SARIMA (58.2%) still lost to Prophet but where Prophet also struggled badly (143.8%) — Florida's Medicaid and hurricane-related spending made both models unreliable.

In [ ]:
rows = []
for r in state_results:
    rows.append({
        'State':     r.get('state_name', r['series'])[:25],
        'Model':     r['model'],
        'MAE ($B)':  round(r['MAE'] / 1e9, 2),
        'RMSE ($B)': round(r['RMSE'] / 1e9, 2),
        'MAPE (%)':  round(r['MAPE'], 1)
    })

results_df = pd.DataFrame(rows)
print(results_df.sort_values(['State','Model']).to_string(index=False))

**Per-State Results — Prophet wins 8 of 10; SARIMA wins only TX and VA**

**Prophet per-state MAPE:**
- South Carolina: **12.9%** — best state result
- Minnesota: **10.9%** — second best
- Pennsylvania: **14.6%**
- Indiana: **15.6%**
- California: **18.3%**
- New York: **21.9%**
- Illinois: **24.5%**
- Virginia: 31.2% (SARIMA 22.9% — **SARIMA wins**)
- Texas: 32.0% (SARIMA 22.6% — **SARIMA wins**)
- Florida: **143.8%** (SARIMA 58.2% — both models failed, SARIMA less bad)

**SARIMA per-state failures:** SARIMA was dramatically worse than Prophet on most states — Pennsylvania (240.9% SARIMA vs 14.6% Prophet), Illinois (100% SARIMA vs 24.5% Prophet), New York (96% SARIMA vs 21.9% Prophet), Indiana (84.6% SARIMA vs 15.6% Prophet). The same differencing problem that caused SARIMA to fail at the total level compounds at the individual state level.

**Florida anomaly:** Both models failed on Florida (143.8% and 58.2%). Florida received disproportionate COVID relief through Medicaid enrollment spikes and hurricane disaster relief in FY2022–2023, creating spending shocks that neither historical trend model could anticipate. This should be flagged in the dashboard.

In [ ]:
# 2×2 grid: top 4 states
top4 = top10_codes[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, code in zip(axes.flat, top4):
    sname = state_names.get(code, code)
    sub   = df[df['geo_code'] == code].sort_values('ds')

    ax.plot(sub['ds'], sub['obligated_amount'], 'o-', ms=2, lw=1.2,
            label='Actual', color='steelblue')

    p_res = next((r for r in state_results if r['series'] == code and r['model'] == 'Prophet'), None)
    s_res = next((r for r in state_results if r['series'] == code and r['model'] == 'SARIMA'), None)

    if p_res:
        ax.plot(p_res['ds'], p_res['pred'], 's--', ms=3, lw=1.2,
                label=f'Prophet ({p_res["MAPE"]:.1f}%)', color='orange')
    if s_res:
        mape_label = f'{s_res["MAPE"]:.1f}%' if s_res['MAPE'] < 1e5 else 'diverged'
        ax.plot(s_res['ds'], s_res['pred'], '^--', ms=3, lw=1.2,
                label=f'SARIMA ({mape_label})', color='green')

    ax.axvline(pd.Timestamp('2021-10-01'), color='red', lw=0.8, ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(billions))
    ax.set_title(f'{code} — {sname}', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Prophet vs SARIMA — Top 4 States (Test Period FY2021–2024)', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**Top 4 states — CA, FL, TX, NY (the 16-quarter test covers COVID peak through FY2024)**

- **California** ($5.6T): Prophet 18.3% vs SARIMA 69.6% — Prophet wins clearly. California's Social Security, Medicare, and defense contract spending grew steadily post-COVID. SARIMA over-differenced and lost the trend.
- **Florida** ($4.9T): Prophet 143.8% vs SARIMA 58.2% — SARIMA is less bad but both failed. Florida's per-quarter spending dropped sharply from FY2021 COVID peaks (enhanced Medicaid, economic relief) to more normal levels in FY2022–2024, creating a structural break neither model anticipated.
- **Texas** ($4.1T): Prophet 32.0% vs **SARIMA 22.6%** — SARIMA wins. Texas defense and energy-sector contract spending followed a regular growth pattern that SARIMA's AR structure tracked better than Prophet's flexible trend.
- **New York** ($3.3T): Prophet 21.9% vs SARIMA 96.0% — Prophet wins. New York's Medicaid and social services funding accelerated through FY2021 and then normalized; Prophet handled this transition better than SARIMA.

## 7. Per-Capita Analysis

In [ ]:
# Per-capita spending by state — Q4 (full-year cumulative) comparison
q4_data = df[df['quarter'] == 4].copy()
q4_data['per_capita'] = q4_data['obligated_amount'] / q4_data['population']

# Latest full year available: FY2024 Q4
latest_fy = q4_data['fy'].max()
latest = q4_data[q4_data['fy'] == latest_fy].sort_values('per_capita', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
top_pc = latest.head(20)
bars = ax.barh(top_pc['geo_name'], top_pc['per_capita'], color='steelblue')
ax.set_xlabel(f'Per Capita Federal Spending (FY{latest_fy} cumulative)')
ax.set_title(f'Top 20 States by Per-Capita Federal Spending — FY{latest_fy}')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.invert_yaxis()
plt.tight_layout(); plt.show()

print(f'\nPer-capita spending — FY{latest_fy} Q4 (full year):')
print(latest[['geo_code','geo_name','obligated_amount','population','per_capita']]
      .head(15).assign(
          obligated_amount=lambda x: (x['obligated_amount']/1e9).round(1),
          per_capita=lambda x: x['per_capita'].round(0).astype(int)
      ).rename(columns={'obligated_amount':'Total ($B)','per_capita':'$/person'})
      .to_string(index=False))

**Per-Capita Findings — DC, North Dakota, and territories lead; large states rank low**

FY2024 per-capita federal spending (Q4 cumulative):
- **DC**: $37,964/person — federal agency concentration, virtually every federal dollar spent here by definition
- **North Dakota**: $30,451/person — surprisingly high; driven by agricultural commodity subsidies (USDA programs) and military installations (Minot AFB, Grand Forks AFB)
- **U.S. Virgin Islands**: $19,349/person — territories receive disproportionate mandatory program funding relative to small populations
- **Guam**: $12,000/person — large military presence (Andersen AFB, Naval Base Guam)
- **Minnesota**: $11,015/person; **Kentucky**: $10,736/person — high per-capita from major federal health program administration centers and Social Security processing hubs

Large states rank near the bottom per-capita despite receiving the most total dollars: California, Texas, and New York have populations in the tens of millions that dilute per-person figures. This chart is the most policy-relevant output for the dashboard geography page.

## 8. Model Comparison

In [ ]:
total_comparison = pd.DataFrame([
    {'Model': 'Prophet',
     'MAE ($B)':  round(prophet_result['MAE']/1e9, 2),
     'RMSE ($B)': round(prophet_result['RMSE']/1e9, 2),
     'MAPE (%)':  round(prophet_result['MAPE'], 1)},
    {'Model': 'SARIMA',
     'MAE ($B)':  round(sarima_result['MAE']/1e9, 2),
     'RMSE ($B)': round(sarima_result['RMSE']/1e9, 2),
     'MAPE (%)':  round(sarima_result['MAPE'], 1)},
    {'Model': 'XGBoost',
     'MAE ($B)':  round(xgb_total_result['MAE']/1e9, 2),
     'RMSE ($B)': round(xgb_total_result['RMSE']/1e9, 2),
     'MAPE (%)':  round(xgb_total_result['MAPE'], 1)},
])
print('=== Total US Geographic Spending — Model Comparison ===')
print(total_comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['orange', 'green', 'purple']
for ax, col in zip(axes, ['MAE ($B)', 'RMSE ($B)', 'MAPE (%)']):
    ax.bar(total_comparison['Model'], total_comparison[col], color=colors)
    ax.set_title(col)
    for i, v in enumerate(total_comparison[col]):
        ax.text(i, v * 1.01, str(v), ha='center', fontsize=9)
plt.suptitle('Geography Forecasting — Model Comparison', y=1.02)
plt.tight_layout()
plt.show()

**Final Scorecard — Prophet wins; SARIMA fails; biggest finding of the project**

| Model   | MAE      | RMSE     | MAPE    |
|---------|----------|----------|---------|
| Prophet | $158B    | $194B    | **13.7%** ✓ |
| XGBoost | $257B    | $373B    | 21.5%   |
| SARIMA  | $673B    | $766B    | **58.3% ✗** |

**The most important finding across all five notebooks:** Model performance is determined by data structure, not data size.
- SARIMA's double differencing was perfectly suited to **cumulative YTD data** (hierarchical notebooks 02–04) and achieved 3.8% MAPE
- The same configuration completely fails on **quarterly spending data** (geography), reaching 58.3% MAPE
- Prophet is the opposite: struggled on cumulative YTD (30% MAPE) but excels on quarterly spending (13.7% MAPE)

**Dashboard model assignment for geography:**
- Total US line: **Prophet** (13.7%)
- Per-state: **Prophet** for 8 of 10 states; **SARIMA** only for Texas and Virginia
- Flag **Florida** as "high uncertainty" — both models fail
- **Do not use SARIMA** for any geography series

## 9. Save Forecasts

In [ ]:
# 1. Total US predictions
total_preds = pd.DataFrame({
    'ds':      test_total['ds'].values,
    'fy':      test_total['fy'].values,
    'quarter': test_total['quarter'].values,
    'actual':  test_total['obligated_amount'].values,
    'prophet': prophet_result['pred'],
    'sarima':  sarima_result['pred'],
    'xgboost': xgb_total['xgb_pred'].values,
})
total_preds.to_csv(FORECAST_DIR / 'geography_total_predictions.csv', index=False)

# 2. Per-state predictions (top 10)
state_pred_rows = []
for r in state_results:
    for ds, pred, true in zip(r['ds'], r['pred'], r['true']):
        state_pred_rows.append({
            'ds': ds, 'geo_code': r['series'],
            'state_name': r.get('state_name', ''),
            'model': r['model'], 'actual': true, 'predicted': pred
        })
pd.DataFrame(state_pred_rows).to_csv(
    FORECAST_DIR / 'geography_per_state_predictions.csv', index=False)

# 3. Per-capita data for dashboard map
pc_out = q4_data[['geo_code','geo_name','fy','obligated_amount','population','per_capita']]
pc_out.to_csv(FORECAST_DIR / 'geography_per_capita_annual.csv', index=False)

# 4. Model metrics
total_comparison.to_csv(FORECAST_DIR / 'geography_model_metrics.csv', index=False)

print('Saved to', FORECAST_DIR)
for f in sorted(FORECAST_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

Four files saved — total predictions, per-state predictions (top 10), per-capita annual data for every state/year (dashboard choropleth map), and model metrics. The per-capita file (`geography_per_capita_annual.csv`) is new compared to notebooks 02–04 and powers the map visualization on the Geography dashboard page without requiring any computation at load time. All geography forecast files use the same `data/forecasts/` folder as the hierarchical notebooks for consistent dashboard loading.